In [5]:
print("Hello World")

Hello World


In [11]:
import pandas as pd
from sklearn.datasets import fetch_openml

In [32]:
titanic = fetch_openml(name="titanic", version=1, as_frame= True)
data = pd.DataFrame(titanic.data)
data.head()
data.count()

c:\ProgramData\anaconda3\Lib\site-packages\sklearn\datasets\_openml.py:968: FutureWarning: The default value of `parser` will change from `'liac-arff'` to `'auto'` in 1.4. You can set `parser='auto'` to silence this warning. Therefore, an `ImportError` will be raised from 1.4 if the dataset is dense and pandas is not installed. Note that the pandas parser may return different data types. See the Notes Section in fetch_openml's API doc for details.
  warn(
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\datasets\_arff_parser.py:200: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  frame = pd.concat(dfs, ignore_index=True)


pclass       1309
name         1309
sex          1309
age          1046
sibsp        1309
parch        1309
ticket       1309
fare         1308
cabin         295
embarked     1307
boat          486
body          121
home.dest     745
dtype: int64

In [2]:
import pandas as pd
import numpy as np
import re 
from sklearn.preprocessing import LabelEncoder

In [15]:
titanic = fetch_openml(name="titanic", version=1, as_frame= True)
data = pd.DataFrame(titanic.data)
print("Original data head:")
print(data.head())
print("\nOriginal data info:")
data.info()


Original data head:
   pclass                                             name     sex      age  \
0     1.0                    Allen, Miss. Elisabeth Walton  female  29.0000   
1     1.0                   Allison, Master. Hudson Trevor    male   0.9167   
2     1.0                     Allison, Miss. Helen Loraine  female   2.0000   
3     1.0             Allison, Mr. Hudson Joshua Creighton    male  30.0000   
4     1.0  Allison, Mrs. Hudson J C (Bessie Waldo Daniels)  female  25.0000   

   sibsp  parch  ticket      fare    cabin embarked  boat   body  \
0    0.0    0.0   24160  211.3375       B5        S     2    NaN   
1    1.0    2.0  113781  151.5500  C22 C26        S    11    NaN   
2    1.0    2.0  113781  151.5500  C22 C26        S  None    NaN   
3    1.0    2.0  113781  151.5500  C22 C26        S  None  135.0   
4    1.0    2.0  113781  151.5500  C22 C26        S  None    NaN   

                         home.dest  
0                     St Louis, MO  
1  Montreal, PQ / Ches

c:\ProgramData\anaconda3\Lib\site-packages\sklearn\datasets\_openml.py:968: FutureWarning: The default value of `parser` will change from `'liac-arff'` to `'auto'` in 1.4. You can set `parser='auto'` to silence this warning. Therefore, an `ImportError` will be raised from 1.4 if the dataset is dense and pandas is not installed. Note that the pandas parser may return different data types. See the Notes Section in fetch_openml's API doc for details.
  warn(
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\datasets\_arff_parser.py:200: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  frame = pd.concat(dfs, ignore_index=True)


In [33]:
# 1. Check for missing values and handle them.
print("\nMissing values before handling:")
print(data.isnull().sum())
# 'Age' - Fill with median
data['age'].fillna(data['age'].median(), inplace=True)
# 'Embarked' - Fill with mode
data['embarked'].fillna(data['embarked'].mode()[0], inplace=True)
# 'Fare' - Fill with median (though often fare might not have many missing values, this is a general approach)
data['fare'].fillna(data['fare'].median(), inplace=True)
# 'Cabin' has a lot of missing values, often dropping or creating a 'Missing' category is best.
# For this exercise, let's create a 'Missing' category for simplicity.
data['cabin'].fillna('Missing', inplace=True)
print("\nMissing values after handling:")
print(data.isnull().sum())


Missing values before handling:
pclass          0
name            0
sex             0
age           263
sibsp           0
parch           0
ticket          0
fare            1
cabin        1014
embarked        2
boat          823
body         1188
home.dest     564
dtype: int64

Missing values after handling:
pclass          0
name            0
sex             0
age             0
sibsp           0
parch           0
ticket          0
fare            0
cabin           0
embarked        0
boat          823
body         1188
home.dest     564
dtype: int64


In [23]:
# 2. Check for duplicate rows and remove them.
print(f"\nNumber of duplicate rows before removal: {data.duplicated().sum()}")
data.drop_duplicates(inplace=True)
print(f"Number of duplicate rows after removal: {data.duplicated().sum()}")


Number of duplicate rows before removal: 0
Number of duplicate rows after removal: 0


In [25]:
# 3. Inspect the "Sex" column for inconsistencies and standardize values.
print("\nUnique values in 'sex' before standardization:")
print(data['sex'].unique())
# Assuming 'female' and 'male' are the only expected values and are already consistent.
# If there were variations (e.g., 'F', 'M', 'woman', 'man'), you'd map them here.
data['sex'] = data['sex'].replace({'male': 'male', 'female': 'female'})
print("Unique values in 'sex' after standardization (if any changes were needed):")
print(data['sex'].unique())



Unique values in 'sex' before standardization:
['female', 'male']
Categories (2, object): ['female', 'male']
Unique values in 'sex' after standardization (if any changes were needed):
['female', 'male']
Categories (2, object): ['female', 'male']


In [26]:
# 4. Detect and handle outliers in the "Age" and "Fare" columns.
# A common method is using the IQR (Interquartile Range) method.
def remove_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    # Cap values rather than removing rows to avoid losing too much data
    df[column] = np.where(df[column] > upper_bound, upper_bound, df[column])
    df[column] = np.where(df[column] < lower_bound, lower_bound, df[column])
    return df
print("\nAge and Fare description before outlier handling:")
print(data[['age', 'fare']].describe())
data = remove_outliers_iqr(data, 'age')
data = remove_outliers_iqr(data, 'fare')
print("\nAge and Fare description after outlier handling (capping):")
print(data[['age', 'fare']].describe())


Age and Fare description before outlier handling:
               age         fare
count  1309.000000  1309.000000
mean     29.503183    33.281086
std      12.905246    51.741500
min       0.166700     0.000000
25%      22.000000     7.895800
50%      28.000000    14.454200
75%      35.000000    31.275000
max      80.000000   512.329200

Age and Fare description after outlier handling (capping):
               age         fare
count  1309.000000  1309.000000
mean     29.187548    24.279696
std      11.972308    20.789492
min       2.500000     0.000000
25%      22.000000     7.895800
50%      28.000000    14.454200
75%      35.000000    31.275000
max      54.500000    66.343800


In [27]:
# 5. Create a new "Family_Size" feature based on "SibSp" and "Parch".
data['family_size'] = data['sibsp'] + data['parch'] + 1 # +1 for the passenger themselves
print("\n'Family_Size' column added:")
print(data[['sibsp', 'parch', 'family_size']].head())


'Family_Size' column added:
   sibsp  parch  family_size
0    0.0    0.0          1.0
1    1.0    2.0          4.0
2    1.0    2.0          4.0
3    1.0    2.0          4.0
4    1.0    2.0          4.0


In [28]:
# 6. Convert the "Embarked" column to numerical values using label encoding.
le = LabelEncoder()
data['embarked_encoded'] = le.fit_transform(data['embarked'])
print("\n'Embarked' and 'embarked_encoded' comparison:")
print(data[['embarked', 'embarked_encoded']].head())
print("Encoded 'embarked' classes:", le.classes_)


'Embarked' and 'embarked_encoded' comparison:
  embarked  embarked_encoded
0        S                 2
1        S                 2
2        S                 2
3        S                 2
4        S                 2
Encoded 'embarked' classes: ['C' 'Q' 'S']


In [ ]:
# 7. If a "Ticket_Purchase_Date" column existed, convert it to datetime and extract year/month.
# (This column does not exist in the Titanic dataset, so this part is illustrative).
# Example if it existed:
# data['Ticket_Purchase_Date'] = pd.to_datetime(data['Ticket_Purchase_Date'])
# data['Ticket_Year'] = data['Ticket_Purchase_Date'].dt.year
# data['Ticket_Month'] = data['Ticket_Purchase_Date'].dt.month
# print("\n'Ticket_Year' and 'Ticket_Month' extracted (if 'Ticket_Purchase_Date' existed):")
# print(data[['Ticket_Purchase_Date', 'Ticket_Year', 'Ticket_Month']].head())


In [30]:
# 8. Clean the "Name" column by removing titles and converting to lowercase.
def clean_name(name):
    # Extract titles (e.g., Mr., Mrs., Miss, Master, etc.)
    title_search = re.search(' ([A-Za-z]+)\.', name)
    if title_search:
        title = title_search.group(1)
    else:
        title = ""
    # Remove titles and non-alphabetic characters, convert to lowercase
    clean_name = re.sub(r'[^a-zA-Z\s]', '', name).lower()
    # You might want to keep or standardize titles for another feature,
    # but the current instruction is to remove them for a 'cleaned name'.
    return clean_name.replace(title.lower(), '').strip()
data['cleaned_name'] = data['name'].apply(clean_name)
print("\nOriginal 'name' and 'cleaned_name' comparison:")
print(data[['name', 'cleaned_name']].head())


Original 'name' and 'cleaned_name' comparison:
                                              name  \
0                    Allen, Miss. Elisabeth Walton   
1                   Allison, Master. Hudson Trevor   
2                     Allison, Miss. Helen Loraine   
3             Allison, Mr. Hudson Joshua Creighton   
4  Allison, Mrs. Hudson J C (Bessie Waldo Daniels)   

                               cleaned_name  
0                   allen  elisabeth walton  
1                    allison  hudson trevor  
2                    allison  helen loraine  
3          allison  hudson joshua creighton  
4  allison  hudson j c bessie waldo daniels  


In [31]:
# 9. Bin the "Age" column into categories like 'Child', 'Teen', 'Adult', and 'Senior'.
bins = [0, 12, 18, 60, np.inf]
labels = ['Child', 'Teen', 'Adult', 'Senior']
data['age_group'] = pd.cut(data['age'], bins=bins, labels=labels, right=False)
print("\n'Age' and 'age_group' comparison:")
print(data[['age', 'age_group']].head())
print("\nFinal data head after all processing steps:")
print(data.head())
print("\nFinal data info after all processing steps:")
data.info()


'Age' and 'age_group' comparison:
    age age_group
0  29.0     Adult
1   2.5     Child
2   2.5     Child
3  30.0     Adult
4  25.0     Adult

Final data head after all processing steps:
   pclass                                             name     sex   age  \
0     1.0                    Allen, Miss. Elisabeth Walton  female  29.0   
1     1.0                   Allison, Master. Hudson Trevor    male   2.5   
2     1.0                     Allison, Miss. Helen Loraine  female   2.5   
3     1.0             Allison, Mr. Hudson Joshua Creighton    male  30.0   
4     1.0  Allison, Mrs. Hudson J C (Bessie Waldo Daniels)  female  25.0   

   sibsp  parch  ticket     fare    cabin embarked  boat   body  \
0    0.0    0.0   24160  66.3438       B5        S     2    NaN   
1    1.0    2.0  113781  66.3438  C22 C26        S    11    NaN   
2    1.0    2.0  113781  66.3438  C22 C26        S  None    NaN   
3    1.0    2.0  113781  66.3438  C22 C26        S  None  135.0   
4    1.0    2.0  113

Explanation of Changes:

Loading Data: Assumed fetch_openml is intended for loading the Titanic dataset, as indicated by your image.
Missing Values:
Age: Filled with the median as age distribution can be skewed.
Embarked: Filled with the mode (most frequent value) as it's a categorical column.
Fare: Filled with the median to handle potential skewness from very expensive tickets.
Cabin: This column typically has a very high number of missing values. Simply filling with the mode or median isn't appropriate. Here, I've filled it with the string 'Missing' to denote the absence of information, which can be useful as a categorical feature.
Outliers: Implemented a function remove_outliers_iqr that uses the Interquartile Range (IQR) method. Instead of removing rows (which can lead to data loss), it caps the outlier values at the upper and lower bounds derived from the IQR. This helps retain data while mitigating the impact of extreme values.
Family Size: Created family_size by summing sibsp (siblings/spouses aboard) and parch (parents/children aboard), and adding 1 for the passenger themselves.
Embarked Encoding: Used LabelEncoder from sklearn.preprocessing to convert the categorical 'embarked' column into numerical values. You can see the mapping with le.classes_.
Ticket Purchase Date: Included a commented-out section to show how you would handle this if the column existed, as it's a common text book exercise.
Name Cleaning: The clean_name function uses a regular expression to extract any title (like Mr., Mrs., etc.) and then removes non-alphabetic characters and titles from the name, converting it to lowercase. This helps standardize names.
Age Binning: Used pd.cut to categorize ages into predefined groups (Child, Teen, Adult, Senior). The right=False argument means the lower bound of the bin is inclusive.

git add FILENAME
git commit -m "MESSAGE HERE"
git push